## 1. Extract cblaster Results


In [ ]:
import pandas as pd
import re

SUMMARY = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\summary.csv'
BINARY  = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\supplement.csv'
OUTPUT  = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\binary_gene.csv'

# ============ Step 1: 解析 summary.csv 建字典 ============
mapping = {}        # (organism, scaffold, query) -> protein_id
organism = None
scaffold = None

with open(SUMMARY) as f:
    lines = [l.rstrip() for l in f]

for i, line in enumerate(lines):
    if not line:
        continue
    nxt = lines[i+1].rstrip() if i+1 < len(lines) else ''
    # organism 行：下一行全是 =
    if re.fullmatch(r'=+', nxt) and not re.fullmatch(r'=+', line) \
            and 'cblaster' not in line.lower():
        organism = line
        continue
    # scaffold 行：下一行全是 -
    if re.fullmatch(r'-+', nxt) and not re.fullmatch(r'-+', line):
        scaffold = line
        continue
    # 基因行：query|locus,protein_id,...
    if '|' in line and ',' in line and organism and scaffold:
        parts = line.split(',')
        mapping[(organism, scaffold, parts[0])] = parts[1]

print(f"summary 解析完成，共 {len(mapping)} 条映射")

# ============ Step 2: 读 binary.csv，替换 1 ============
df = pd.read_csv(BINARY, sep=',')
meta_cols = ['Organism', 'Scaffold', 'Start', 'End', 'Score']
gene_cols = [c for c in df.columns if c not in meta_cols]

# 关键：基因列原本是 int，要转 object 才能存字符串
for col in gene_cols:
    df[col] = df[col].astype(object)

n_replaced = 0
n_missing = 0
for idx, row in df.iterrows():
    org = row['Organism']
    sca = row['Scaffold']
    for col in gene_cols:
        if row[col] == 1:
            key = (org, sca, col)
            if key in mapping:
                df.at[idx, col] = mapping[key]
                n_replaced += 1
            else:
                df.at[idx, col] = 'MISSING'   # 找不到的标记一下
                n_missing += 1
        else:
            df.at[idx, col] = ''   # 0 改成空；想保留 0 就改成 '0'

print(f"替换了 {n_replaced} 个 1 → protein_id")
print(f"找不到映射: {n_missing} 个")

df.to_csv(OUTPUT, sep=',', index=False)
print(f"输出: {OUTPUT}")

## 2. Extract Gene-Cluster Regions from GenBank Files


In [ ]:
"""根据 binary.csv 提取 gbff 文件里对应区段"""
import pandas as pd
import os
from Bio import SeqIO

BINARY = '/public/home/luyt/Data/Ral/fadgdh/absence_phylogeny/cblaster_flank/extract_gbff/binary_gene.csv'
GBFF_DIR = '/public/home/luyt/Data/Ral/allral_genome/1267/1267/file'
OUT_DIR = '/public/home/luyt/Data/Ral/fadgdh/absence_phylogeny/cblaster_flank/extract_gbff/gbff'
os.makedirs(OUT_DIR, exist_ok=True)

df = pd.read_csv(BINARY, sep=',')

n_ok = 0
n_missing_file = 0
n_missing_scaffold = 0

for idx, row in df.iterrows():
    gc = row['Organism']
    sca = row['Scaffold']
    start = int(row['Start'])
    end = int(row['End'])

    gbff = os.path.join(GBFF_DIR, f"{gc}.gbff")
    if not os.path.isfile(gbff):
        print(f"[缺文件] {gbff}")
        n_missing_file += 1
        continue

    found = False
    for rec in SeqIO.parse(gbff, 'genbank'):
        rec_id = rec.id
        if rec_id == sca:
            # cblaster 的 0-based 坐标直接用于 slice
            sub = rec[start:end]
            sub.id = f"{gc}_{sca}_{start}_{end}"
            sub.description = f"extracted from {sca} {start}-{end}"
            out = os.path.join(OUT_DIR, f"{gc}.gbff")
            SeqIO.write([sub], out, 'genbank')
            found = True
            n_ok += 1
            break

    if not found:
        print(f"[找不到 scaffold] {gc} / {sca}")
        n_missing_scaffold += 1

print(f"\n成功: {n_ok}")
print(f"缺 gbff 文件: {n_missing_file}")
print(f"找不到 scaffold: {n_missing_scaffold}")
print(f"输出目录: {OUT_DIR}")

## 3. Replace Protein IDs with Locus Tags and Fill in Pseudogene Locus Tags

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
把基因簇表格里的蛋白号 (WP_...) 替换成对应基因组的 locus_tag，
并为空着的假基因单元格按 /product 名字补上 locus_tag。
"""

import csv
import glob
import os
import sys

from Bio import SeqIO

# ==================== 在这里改路径 ====================
TABLE  = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\type.csv"      # 输入 TSV 表格
GBFF_DIR = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\final_gbff"      # 放 gbff 的文件夹，文件名 = Organism列的GCF号，如 GCF_000009125.1.gbff
OUTPUT = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\type_with_locus_tags.csv"          # 输出 TSV
# ====================================================

# ---- 每个基因列(取 | 前的角色名)对应的 /product 名字 ----
PRODUCT_MAP = {
    "cytochromeC":    ["c-type cytochrome", "cytochrome c"],
    "oxidoreductase": ["GMC family oxidoreductase"],
    "small_subunit":  ["sugar dehydrogenase complex small subunit"],
    "left1":          ["AsmA family protein"],
    "right1":         ["glycoside hydrolase family 3 protein"],
    "right2":         ["pyridoxamine 5'-phosphate oxidase"],
}

EXTS = (".gbff", ".gbf", ".gb", ".gbk", ".genbank")


def build_file_index(gbff_dir):
    """文件名(去扩展名) -> 路径，并加一个去版本号的兜底键。"""
    idx = {}
    files = []
    for ext in EXTS:
        files += glob.glob(os.path.join(gbff_dir, "**", "*" + ext), recursive=True)
    for fp in sorted(set(files)):
        stem = os.path.basename(fp)
        for ext in EXTS:
            if stem.endswith(ext):
                stem = stem[: -len(ext)]
                break
        idx.setdefault(stem, fp)                # 如 GCF_000009125.1
        idx.setdefault(stem.split(".")[0], fp)  # 去版本号兜底 GCF_000009125
    return idx


def locus_table(records):
    """records 列表内: locus_tag -> {protein_id, products(set), pseudo, start}"""
    d = {}
    for rec in records:
        for f in rec.features:
            if f.type not in ("gene", "CDS"):
                continue
            lt = f.qualifiers.get("locus_tag", [None])[0]
            if not lt:
                continue
            e = d.setdefault(lt, {"protein_id": None, "products": set(),
                                  "pseudo": False, "start": int(f.location.start)})
            if "protein_id" in f.qualifiers:
                e["protein_id"] = f.qualifiers["protein_id"][0]
            for p in f.qualifiers.get("product", []):
                e["products"].add(p)
            if "pseudo" in f.qualifiers or "pseudogene" in f.qualifiers:
                e["pseudo"] = True
    return d


def pick_records(all_recs, scaffold):
    """文件内多条记录时，优先取与 Scaffold 一致的那条；取不到就用全部。"""
    sc = scaffold.strip()
    hit = [r for r in all_recs
           if sc in {r.id, r.name, r.id.split(".")[0]}
           or sc.split(".")[0] == r.id.split(".")[0]]
    return hit or all_recs


def product_matches(products, wanted_lower):
    prods = [p.lower() for p in products]
    return any(w == p or w in p for w in wanted_lower for p in prods)


def main():
    if not os.path.isdir(GBFF_DIR):
        sys.exit(f"找不到文件夹: {GBFF_DIR}")
    file_idx = build_file_index(GBFF_DIR)
    if not file_idx:
        sys.exit("文件夹里没找到 gbff 文件")

    with open(TABLE, newline="") as f:
        rows = list(csv.reader(f, delimiter=","))
    if not rows:
        sys.exit("表格是空的")

    header = rows[0]
    gene_cols = [i for i, h in enumerate(header) if "|" in h]   # 9 个基因列
    roles = {i: header[i].split("|")[0] for i in gene_cols}
    organism_i = header.index("Organism")
    scaffold_i = header.index("Scaffold")

    cache = {}   # 路径 -> 解析好的 records，避免重复解析
    log = []
    n_repl = n_fill = 0

    for r_i in range(1, len(rows)):
        row = rows[r_i]
        if len(row) < len(header):
            row += [""] * (len(header) - len(row))
            rows[r_i] = row

        gcf = row[organism_i].strip()
        fp = file_idx.get(gcf) or file_idx.get(gcf.split(".")[0])
        if not fp:
            log.append(f"[行{r_i}] 找不到文件 {gcf}.gbff，整行跳过")
            continue

        if fp not in cache:
            cache[fp] = list(SeqIO.parse(fp, "genbank"))
        recs = pick_records(cache[fp], row[scaffold_i])

        lt = locus_table(recs)
        pid2lt = {v["protein_id"]: k for k, v in lt.items() if v["protein_id"]}
        used = set()

        # 第一步：WP -> locus_tag
        for i in gene_cols:
            val = row[i].strip()
            if val.startswith("WP_"):
                if val in pid2lt:
                    row[i] = pid2lt[val]
                    used.add(pid2lt[val])
                    n_repl += 1
                else:
                    log.append(f"[行{r_i}] {gcf}: {val} 不在该样本里，保留原值")

        # 第二步：空格(假基因) -> 按 product 补 locus_tag
        for i in gene_cols:
            if row[i].strip():
                continue
            role = roles[i]
            if role not in PRODUCT_MAP:
                continue
            wanted = [w.lower() for w in PRODUCT_MAP[role]]
            cands = [(v["start"], k) for k, v in lt.items()
                     if not v["protein_id"] and k not in used
                     and product_matches(v["products"], wanted)]
            if not cands:
                log.append(f"[行{r_i}] {gcf}: {role} 为空且未找到匹配 /product 的假基因")
                continue
            cands.sort()
            row[i] = cands[0][1]
            used.add(cands[0][1])
            n_fill += 1
            if len(cands) > 1:
                log.append(f"[行{r_i}] {gcf}: {role} 有多个候选 {[c[1] for c in cands]}，取最靠前的")

    with open(OUTPUT, "w", newline="") as f:
        csv.writer(f, delimiter=",").writerows(rows)

    if log:
        sys.stderr.write("\n".join(log) + "\n")
    print(f"完成 -> {OUTPUT}  (替换 {n_repl} 个蛋白号，补全 {n_fill} 个假基因)")


if __name__ == "__main__":
    main()

## 4. Plot Gene-Cluster Organization by Phylotype


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""每个 phylotype 单独出一个 PDF，簇内按 type 排序画 cluster 图。
表格里已是 locus_tag，假基因按所属基因列的颜色画并保留 /// 斜线标记。"""

from pygenomeviz import GenomeViz
from Bio import SeqIO
import matplotlib.pyplot as plt
import pandas as pd
import os, glob

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype']  = 42
# 用一个常规无衬线字体，避免 AI 端缺字
matplotlib.rcParams['font.family'] = 'sans-serif'
matplotlib.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']

# ==================== 在这里改路径 ====================
CSV      = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\type_with_locus_tags.csv'
GBFF_DIR = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\final_gbff'
OUT_DIR  = r'E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\fig\fig2'
# ====================================================
os.makedirs(OUT_DIR, exist_ok=True)

GENE_COLORS = {
    'left3': '#1f77b4', 'left2': '#ff7f0e', 'left1': '#2ca02c',
    'cytochromeC': '#ffca3a', 'oxidoreductase': '#9467bd',
    'small_subunit': '#8c564b', 'right1': '#e377c2',
    'right2': '#43aa8b', 'right3': '#bcbd22',
}
ANCHOR_PRIORITY = ['left1', 'left3', 'right3']   # 判方向用的锚基因
EMPTY = ('', '0', 'missing', 'nan')


def read_table(path):
    """自动识别分隔符(制表符/逗号)，空格保留为空字符串。"""
    with open(path, encoding='utf-8-sig') as f:
        first = f.readline()
    sep = '\t' if first.count('\t') >= first.count(',') else ','
    return pd.read_csv(path, sep=sep, encoding='utf-8-sig',
                       dtype=str, keep_default_na=False)


def type_key(t):
    t = str(t).strip()
    try:
        return (0, int(t))
    except ValueError:
        return (1, t)


def role_strand(rec, role, locus_to_query):
    for feat in rec.features:
        if feat.type not in ('CDS', 'gene'):
            continue
        locus = feat.qualifiers.get('locus_tag', [''])[0]
        if locus and locus_to_query.get(locus) == role:
            return feat.location.strand
    return None


def load_record(gbff, scaffold):
    """取与 Scaffold 一致的记录；取不到就用第一条。"""
    recs = list(SeqIO.parse(gbff, 'genbank'))
    if not recs:
        return None
    sc = str(scaffold).strip()
    for r in recs:
        if sc in {r.id, r.name} or sc.split('.')[0] == r.id.split('.')[0]:
            return r
    return recs[0]


def features_by_locus(rec):
    """每个 locus 只取一个 feature(优先 CDS)，并合并 gene/CDS 上的 pseudo 标记。
    这样即使假基因只写成 gene feature(无 CDS) 也能被画出来。"""
    by_locus, no_locus = {}, []
    for feat in rec.features:
        if feat.type not in ('CDS', 'gene'):
            continue
        pseudo = 'pseudo' in feat.qualifiers or 'pseudogene' in feat.qualifiers
        locus = feat.qualifiers.get('locus_tag', [''])[0]
        if not locus:
            if feat.type == 'CDS':
                no_locus.append((feat, pseudo))
            continue
        e = by_locus.setdefault(locus, {'feat': None, 'pseudo': False})
        e['pseudo'] = e['pseudo'] or pseudo
        if e['feat'] is None or (e['feat'].type == 'gene' and feat.type == 'CDS'):
            e['feat'] = feat
    items = [(locus, v['feat'], v['pseudo']) for locus, v in by_locus.items()]
    items += [(None, feat, pseudo) for feat, pseudo in no_locus]
    return items


def main():
    df = read_table(CSV)
    gene_cols = [c for c in df.columns if '|' in c]      # 9 个基因列
    df['phylotype'] = df['phylotype'].astype(str).str.strip()

    # locus_tag -> 所属基因列(角色名)
    # 注意：单元格里可能有用 | 分隔的多个 locus_tag —— 同一个基因的假基因被
    # gbff 拆成了多段(如 RS25430|RS26465)，逐个拆开，都映射到同一基因列。
    locus_to_query = {}
    for _, row in df.iterrows():
        for col in gene_cols:
            role = col.split('|')[0]
            for lt in str(row[col]).split('|'):
                lt = lt.strip()
                if lt and lt.lower() not in EMPTY:
                    locus_to_query[lt] = role

    for phylo in sorted(df['phylotype'].unique(), key=type_key):
        sub = df[df['phylotype'] == phylo]

        # ---- 收集该 phylotype 的样本，准备方向 ----
        cands = []   # (排序键, type字符串, 名称, rec, need_reverse)
        for _, row in sub.iterrows():
            org = str(row['Organism']).strip()
            gbff = os.path.join(GBFF_DIR, org + '.gbff')
            if not os.path.exists(gbff):
                hits = glob.glob(os.path.join(GBFF_DIR, org.split('.')[0] + '*.gbff'))
                gbff = hits[0] if hits else None
            if not gbff:
                print(f"  [缺文件] {org}.gbff 找不到，跳过")
                continue

            rec = load_record(gbff, row['Scaffold'])
            if rec is None or len(rec.seq) == 0:
                print(f"  [空记录] {org}，跳过")
                continue

            strand = None
            for a in ANCHOR_PRIORITY:
                strand = role_strand(rec, a, locus_to_query)
                if strand is not None:
                    break
            need_reverse = (strand == -1)

            typ = str(row['loss_type']).strip()
            if typ == '1' or typ=='unknown':          # type1 不画
                continue
            cands.append((type_key(typ), typ, org, rec, need_reverse))

        if not cands:
            print(f"Phylotype {phylo}: 无可画样本")
            continue

        cands.sort(key=lambda x: x[0])   # 按 type 排序

        # ---- 画图 ----
        gv = GenomeViz(fig_track_height=0.5, feature_track_ratio=1.0)
        for idx, (_, typ, name, rec, need_reverse) in enumerate(cands, 1):
            if need_reverse:
                rec = rec.reverse_complement(id=True, name=True, description=True,
                                             features=True, annotations=True)
            track = gv.add_feature_track(f"{idx}. {name} (type{typ})", len(rec.seq))

            for locus, feat, is_pseudo in features_by_locus(rec):
                query = locus_to_query.get(locus) if locus else None
                if is_pseudo:
                    # 底色保持白色，只让 /// 斜线(和边框)用该基因列的颜色
                    color = 'white'
                    hatch = '///'
                    edge = GENE_COLORS.get(query, 'gray')
                elif query:
                    color, hatch, edge = GENE_COLORS.get(query, 'gray'), None, 'black'
                else:
                    color, hatch, edge = 'gray', None, 'gray'

                track.add_feature(
                    int(feat.location.start),
                    int(feat.location.end),
                    strand=feat.location.strand or 1,
                    label='',
                    facecolor=color,
                    edgecolor=edge,
                    hatch=hatch,
                    lw=0.3,
                )

        out_file = os.path.join(OUT_DIR, f'phylotype{phylo}.pdf')
        fig = gv.plotfig()
        fig.savefig(out_file, format='pdf', bbox_inches='tight')
        plt.close(fig)
        print(f"Phylotype {phylo}: {len(cands)} 个 -> {out_file}")


if __name__ == '__main__':
    main()

## 5. Extract Pseudogenes

In [ ]:
import os
import pandas as pd
from Bio import SeqIO

# ============ 配置区 ============
CSV_FILE = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\type.csv"
GBFF_DIR = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\final_gbff"
OUT_DIR  = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\sequence"

# 筛选条件:phylotype -> 允许的 type 列表
FILTER = {
    1: [2,3,4,5,6],
    2: [2,3,4,5,6],
    3: [2,3,4,5,6],
    4: [2,3,4,5,6],
}

# 是否只提取带 /pseudo 的 feature
REQUIRE_PSEUDO = True

# 每个类型 -> (输出文件名, [所有算作这一类的 product 写法,严格匹配])
PRODUCT_GROUPS = {
    "type1": ("type1_cytochrome.txt",
              ["c-type cytochrome", "cytochrome c"]),
    "type2": ("type2_oxidoreductase.txt",
              ["GMC family oxidoreductase"]),
    "type3": ("type3_small_subunit.txt",
              ["sugar dehydrogenase complex small subunit"]),
}
# ================================

os.makedirs(OUT_DIR, exist_ok=True)

# 由分组反查:product 字符串 -> 所属类型 key,方便匹配
prod_to_group = {}
for grp, (fname, prods) in PRODUCT_GROUPS.items():
    for p in prods:
        prod_to_group[p] = grp

# --- 1. 读 csv 并筛样本 ---
df = pd.read_csv(CSV_FILE, sep=None, engine="python")
df["phylotype"] = pd.to_numeric(df["phylotype"], errors="coerce")
df["type"] = pd.to_numeric(df["type"], errors="coerce")

mask = pd.Series(False, index=df.index)
for ph, types in FILTER.items():
    mask |= (df["phylotype"] == ph) & (df["type"].isin(types))
selected = df[mask]
gcf_list = selected["Organism"].astype(str).str.strip().tolist()
print(f"筛中 {len(gcf_list)} 个样本")

# --- 2. 准备输出句柄(按类型,一类一个文件)---
out_files = {grp: open(os.path.join(OUT_DIR, fname), "w")
             for grp, (fname, _) in PRODUCT_GROUPS.items()}
counts = {grp: 0 for grp in PRODUCT_GROUPS}

# --- 3. 逐样本处理 ---
n_nofile = 0
for gcf in gcf_list:
    candidates = [gcf]

    gbff_path = None
    for c in candidates:
        p = os.path.join(GBFF_DIR, f"{c}.gbff")
        if os.path.isfile(p):
            gbff_path = p
            break
    if not gbff_path:
        print(f"[缺 gbff] {gcf}")
        n_nofile += 1
        continue

    for rec in SeqIO.parse(gbff_path, "genbank"):
        for feat in rec.features:
            if feat.type != "CDS":
                continue
            if REQUIRE_PSEUDO and "pseudo" not in feat.qualifiers:
                continue
            prod = feat.qualifiers.get("product", [None])[0]
            grp = prod_to_group.get(prod)               # 查这个 product 属于哪一类
            if grp is not None:                          # 严格匹配到某一类
                nt_seq = feat.extract(rec.seq)           # 自动处理正负链
                out_files[grp].write(f">{gcf}\n{nt_seq}\n")
                counts[grp] += 1

# --- 4. 收尾 ---
for fh in out_files.values():
    fh.close()

print("\n各类型提取数量:")
for grp, (fname, _) in PRODUCT_GROUPS.items():
    print(f"  {counts[grp]:>4}  {fname}")
if n_nofile:
    print(f"\n缺 gbff 文件:{n_nofile} 个")

## 6.Classify Pseudogene Types


In [ ]:
#!/usr/bin/env python3
# ============ 参数：改这里就行 ============
ALN_FILE  = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\macse\mafft_result\type3_small_subunit_mafft.fa"   # MAFFT比对结果文件
REF_NAME  = "GMI1000"                      # 参考序列名
COV_CUT   = 0.5                            # 覆盖度阈值，低于此判“片段/大缺失型”
GAP_CUT   = 30                             # 框内缺失≥此bp(且3倍数)判“框内缺失型”
OUT_CSV   = r"E:\Data\Ralstonia\new_analysis\FADGDH\absence_phylogeny\analysis2\macse\mafft_result\type3_small_subunit_classification.csv"   # 输出文件名
# ==========================================

import csv, itertools

# 密码子表
CT={'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G'}

# ---- 读取fasta ----
seqs={}; order=[]; name=None; buf=[]
for line in open(ALN_FILE):
    line=line.rstrip()
    if line.startswith('>'):
        if name: seqs[name]=''.join(buf)
        name=line[1:].split()[0]; buf=[]; order.append(name)
    else:
        buf.append(line)
if name: seqs[name]=''.join(buf)
seqs={k:v.upper() for k,v in seqs.items()}

# ---- 参考坐标 ----
ref=seqs[REF_NAME]
refcols=[i for i,c in enumerate(ref) if c!='-']   # 参考非gap的列
reflen=len(refcols)                                # 参考去gap长(bp)

# ---- 逐条统计 ----
rows=[]
for n in order:
    s=seqs[n]
    proj=''.join(s[i] for i in refcols)   # 投影到参考坐标
    raw=s.replace('-','')                  # 去gap原始序列
    covered=sum(1 for c in proj if c!='-')
    cov=covered/reflen

    # 最大连续缺失
    best=0; best_start=0; cur=0; start=0
    for i,c in enumerate(proj):
        if c=='-':
            if cur==0: start=i
            cur+=1
            if cur>best: best=cur; best_start=start
        else: cur=0
    gap_run=best; gap_pos=best_start

    # 翻译找内部终止
    prot=''.join(CT.get(raw[i:i+3],'X') for i in range(0,len(raw)-2,3))
    body=prot[:-1] if prot.endswith('*') else prot
    nstop=body.count('*')
    first_stop=body.find('*')+1 if '*' in body else 0
    mod3=len(raw)%3

    # 判定(优先级从上到下)
    if n==REF_NAME:                 typ='Reference (intact)'
    elif cov < COV_CUT:             typ='incomplete'
    elif mod3!=0:                   typ='frameshifted'
    elif nstop>0:                   typ='internal stop'
    elif gap_run>=GAP_CUT:          typ='in-frame deletion'
    else:                           typ='complete'

    rows.append([n,len(raw),round(cov*100,1),mod3,nstop,
                 first_stop if first_stop else '',
                 gap_run,(gap_pos+1) if gap_run else '',typ])

# ---- 写CSV ----
with open(OUT_CSV,'w',newline='') as f:
    w=csv.writer(f)
    w.writerow(['Sequence','UngappedLen_bp','Coverage_%','Len_mod3',
                'InternalStops','FirstStop_aa','MaxDeletion_bp',
                'DeletionStart_refnt','Type'])
    w.writerows(rows)

# ---- 屏幕汇总 ----
from collections import Counter
cnt=Counter(r[-1] for r in rows)
print(f"参考: {REF_NAME}  参考长: {reflen} bp  覆盖度阈值: {COV_CUT*100:.0f}%  共 {len(rows)} 条\n")
print("== 分类计数 ==")
for k,v in cnt.most_common(): print(f"  {k:<14}{v}")
print(f"\n== 异常序列明细 (非功能完整) ==")
print(f"{'序列名':<22}{'长bp':>6}{'覆盖%':>7}{'%3':>4}{'终止':>5}{'首*aa':>7}{'最大缺失bp':>10}{'缺失起点':>9}  类型")
for r in rows:
    if r[-1] in ('complete','Reference (intact)'): continue
    print(f"{r[0]:<22}{r[1]:>6}{r[2]:>7}{r[3]:>4}{r[4]:>5}{str(r[5]):>7}{r[6]:>10}{str(r[7]):>9}  {r[8]}")
print(f"\n完整结果已写入: {OUT_CSV}")